# Comparaison des méthodes

Ce notebook rassemble **tous les dossiers d'un groupe d'une campagne** (même dataset, même architecture, même K) et
compare : chaque méthode pour chaque point de la grille (alpha, lr...), les agents indépendants (alpha = 0) et le
grand modèle seul (`results/reference/`).

Toutes les statistiques sont des **médianes [quantile 5 % ; quantile 95 %] sur les graines**. La comparaison
« graine par graine » (différence appariée) est plus fiable que la comparaison des médianes : deux méthodes
lancées avec la même graine voient les mêmes initialisations et les mêmes données.

Les figures et le tableau sont enregistrés dans `results/<campagne>/<dataset>/<arch>_<K>agents/figures/notebooks/compare_methods/`.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

# --- À régler -------------------------------------------------------------
CODE_DIR = ".."                # racine du dépôt (contient collaborative_learning/ et configs/)
RESULTS_ROOT = "../results"    # dossier des résultats ("../results_quick_test" pour la campagne quick_test)
CAMPAIGN = "alpha_sweep"        # nom dans configs/campaigns/, ou chemin d'un fichier de campagne
DATASET = None                 # None = le premier dataset de la campagne, sinon ex. "cifar10"
K = None                       # None = le premier K de la campagne
REFERENCE = None               # méthode de référence pour les différences appariées (None : grand modèle, sinon agents indépendants)
# ---------------------------------------------------------------------------

sys.path.insert(0, os.path.abspath(CODE_DIR))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from experiments.campaign import Campaign, point_name
from collaborative_learning.evaluation.plots import campaign_groups, entry_name, method_color, method_label

campaign = Campaign(CAMPAIGN, {"out_dir": RESULTS_ROOT})
groups = {g: v for g, v in campaign_groups(campaign).items()
          if (DATASET is None or g[0] == DATASET) and (K is None or g[2] == K) and v["entries"]}
if not groups:
    raise FileNotFoundError(f"Aucun résultat fusionné pour la campagne {campaign.name} dans {os.path.abspath(RESULTS_ROOT)} "
                            f"(python -m experiments.campaign {campaign.name} --merge)")
(DATASET, ARCH, K), group = next(iter(groups.items()))
entries, reference, GROUP_DIR = group["entries"], group["reference"], group["dir"]
print(f"Campagne {campaign.name} : {DATASET}, {K} × {ARCH}  |  {len(entries)} dossier(s) de K agents  |  "
      f"grand modèle : {os.path.relpath(reference['dir'], RESULTS_ROOT) if reference else 'aucun'}")

OUT_DIR = os.path.join(GROUP_DIR, "figures", "notebooks", "compare_methods")
os.makedirs(OUT_DIR, exist_ok=True)
for e in entries:
    print(" ", os.path.relpath(e["dir"], GROUP_DIR))

## Lecture des runs

Une ligne par (méthode, point de la grille, graine). Avec α = 0, il n'y a pas d'interaction : ce sont des agents
**indépendants**, chacun en cross-entropy classique, avec un vote majoritaire.

In [ ]:
def last(history, key):
    values = history.get(key) or []
    return values if isinstance(values, list) else []

BIG_FAMILY = "Grand modèle"
rows, curves = [], []
for e in entries:
    name, family = entry_name(e), method_label(e["method"])
    for r in e["results"]["runs"]:            # K agents, une ligne par graine
        rows.append(dict(method=name, family=family, method_id=e["method"], seed=r["seed"],
                         consensus=r["accuracy_consensus"], ensemble=r["accuracy_mean_ensemble"],
                         agent_median=r["accuracy_agent_median"], agent_best=r["accuracy_agent_best"]))
        curves.append(dict(method=name, seed=r["seed"], val=last(r["history"], "val_accuracy")))
if reference:                                 # grand modèle seul
    sizes = reference["results"].get("model_sizes") or {}
    big = f"Grand modèle (largeur ×{sizes.get('big_width', '?')})"
    for b in reference["results"]["runs"]:
        rows.append(dict(method=big, family=BIG_FAMILY, method_id="reference", seed=b["seed"],
                         consensus=b["accuracy"], ensemble=b["accuracy"], agent_median=b["accuracy"],
                         agent_best=b["accuracy"]))
        curves.append(dict(method=big, seed=b["seed"], val=last(b["history"], "val_accuracy")))

df = pd.DataFrame(rows)
curves_df = pd.DataFrame(curves)
print(f"{df['method'].nunique()} méthodes / points, graines : {sorted(int(s) for s in df['seed'].unique())}")

## Tableau récapitulatif

Médiane [q05 ; q95] sur les graines, triée par accuracy du consensus.
- **consensus** : prédiction du système (stratégie de la config ; vote majoritaire pour α = 0) ;
- **ensemble** : moyenne simple des probabilités des agents ;
- **agent médian / meilleur agent** : un agent seul, comme Zhang et al. évaluent la DML.

In [ ]:
METRICS = {"consensus": "Consensus", "ensemble": "Moyenne simple", "agent_median": "Agent médian", "agent_best": "Meilleur agent"}

def fmt(values):
    v = np.asarray(values, dtype=float)
    return f"{np.median(v):.4f} [{np.quantile(v, 0.05):.4f} ; {np.quantile(v, 0.95):.4f}]"

order = df.groupby("method")["consensus"].median().sort_values(ascending=False).index
table = pd.DataFrame({label: df.groupby("method")[key].apply(fmt) for key, label in METRICS.items()}).loc[order]
table.insert(0, "Famille", df.groupby("method")["family"].first().loc[order])
table["Graines"] = df.groupby("method")["seed"].nunique().loc[order]
table.to_csv(os.path.join(OUT_DIR, "comparison.csv"))
display(table)

## Accuracy par méthode

Point plein : consensus. Point creux : agent médian. Barre : intervalle [q05 ; q95] sur les graines.

In [ ]:
# Couleur par méthode, la même que dans les figures de plot_results ; gris pour le grand modèle
FAMILY_COLORS = {method_label(m): method_color(m) for m in df["method_id"].unique() if m != "reference"}
FAMILY_COLORS[BIG_FAMILY] = "#8a8984"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def style(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelcolor=INK)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)

methods = list(order)[::-1]                    # meilleure méthode en haut
fig, ax = plt.subplots(figsize=(8, 0.45 * len(methods) + 1.4))
for i, m in enumerate(methods):
    sub = df[df["method"] == m]
    color = FAMILY_COLORS.get(sub["family"].iloc[0], MUTED)
    for key, offset, filled in (("consensus", 0.12, True), ("agent_median", -0.12, False)):
        v = sub[key].to_numpy()
        med, lo, hi = np.median(v), np.quantile(v, 0.05), np.quantile(v, 0.95)
        ax.plot([lo, hi], [i + offset] * 2, color=color, linewidth=2, solid_capstyle="round")
        ax.plot(med, i + offset, "o", markersize=8, markeredgewidth=2, markeredgecolor=color,
                markerfacecolor=color if filled else "white", zorder=3)
ax.set_yticks(range(len(methods)), methods)
ax.set_xlabel("accuracy test", color=INK)
style(ax)
handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=8, markeredgewidth=2, markeredgecolor=MUTED,
                      markerfacecolor=f, label=l) for f, l in ((MUTED, "consensus"), ("white", "agent médian"))]
handles += [plt.Line2D([], [], color=c, linewidth=6, label=f) for f, c in FAMILY_COLORS.items()
            if f in set(df["family"])]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_title(f"{ARCH} x{K} sur {DATASET} : médiane [q05 ; q95] sur les graines", color=INK, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy_by_method.png"), dpi=200, bbox_inches="tight")
plt.show()

## Différence appariée avec la référence

Pour chaque graine : accuracy du consensus de la méthode − accuracy de la référence **avec la même graine**.
Une méthode dont tous les points sont à droite de 0 bat la référence sur chaque graine.
Référence par défaut : le grand modèle s'il existe, sinon l'ensemble indépendant.

In [ ]:
ref = REFERENCE
if ref is None:
    for family in (BIG_FAMILY, method_label("independent")):
        names = df.loc[df["family"] == family, "method"].unique()
        if len(names):
            ref = sorted(names)[0]
            break
ref_acc = df[df["method"] == ref].set_index("seed")["consensus"]
others = [m for m in methods if m != ref]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(others) + 1.4))
summary = {}
for i, m in enumerate(others):
    sub = df[df["method"] == m].set_index("seed")
    common = sub.index.intersection(ref_acc.index)
    if not len(common):
        continue
    diff = (sub.loc[common, "consensus"] - ref_acc.loc[common]).to_numpy() * 100
    color = FAMILY_COLORS.get(sub["family"].iloc[0], MUTED)
    jitter = np.linspace(-0.15, 0.15, len(diff)) if len(diff) > 1 else [0.0]
    ax.plot(diff, i + np.asarray(jitter), "o", markersize=8, color=color, markeredgecolor="white", markeredgewidth=2)
    ax.plot(np.median(diff), i, "|", markersize=18, markeredgewidth=2, color=INK)
    summary[m] = (np.median(diff), int((diff > 0).sum()), len(diff))
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_yticks(range(len(others)), others)
ax.set_xlabel(f"écart d'accuracy avec « {ref} » (points de %)", color=INK)
style(ax)
ax.set_title("Différence appariée par graine (trait noir : médiane)", color=INK, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "paired_difference.png"), dpi=200, bbox_inches="tight")
plt.show()
for m, (med, wins, n) in sorted(summary.items(), key=lambda kv: -kv[1][0]):
    print(f"{m:45s} {med:+6.2f} pts (médiane) | meilleure sur {wins}/{n} graines")

## Courbes d'apprentissage

Accuracy de validation par époque (consensus ; le modèle seul pour le grand modèle), médiane sur les graines,
zone = [q05 ; q95]. Une couleur par méthode ; quand une méthode a plusieurs points (alpha, lr...), les traits se
distinguent par le style (plein, tirets, pointillés).

In [ ]:
if len(curves_df) and curves_df["val"].map(len).max() > 0:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    dashes = ["-", "--", ":", "-."]
    seen = {}
    fam = df.groupby("method")["family"].first()
    for m in order:
        sub = curves_df[curves_df["method"] == m]
        if not len(sub):
            continue
        n = min(len(v) for v in sub["val"])
        if n == 0:
            continue
        arr = np.array([v[:n] for v in sub["val"]])
        x = np.arange(1, n + 1)
        family = fam.get(m, "")
        color = FAMILY_COLORS.get(family, MUTED)
        seen[family] = seen.get(family, -1) + 1
        ls = dashes[seen[family] % len(dashes)]
        ax.fill_between(x, np.quantile(arr, 0.05, axis=0), np.quantile(arr, 0.95, axis=0), color=color, alpha=0.12, linewidth=0)
        ax.plot(x, np.median(arr, axis=0), color=color, linestyle=ls, linewidth=2, label=m)
    ax.set_xlabel("époque", color=INK)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylabel("accuracy validation", color=INK)
    style(ax)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    ax.set_title("Courbes d'apprentissage (médiane sur les graines)", color=INK, loc="left")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "validation_curves.png"), dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Pas d'historique de validation dans ces runs.")
print("Figures et tableau :", os.path.abspath(OUT_DIR))